# Notebook 6 — Stage 4: Evaluation
**MAI 656 — Natural Language Processing | Canadian University Dubai | Spring 2026**

This notebook evaluates the fine-tuned `Qwen/Qwen2.5-VL-7B-Instruct` + LoRA adapter on the held-out eval set, reporting:
- **CER** — Character Error Rate (via `jiwer`)
- **WER** — Word Error Rate (via `jiwer`)
- **Exact match rate**
- **Valid JSON rate**
- **Average inference time per sample**

**Reference labels.** Ground-truth transcriptions in `data/eval/eval.jsonl` were generated with **gpt-5.4-mini** during dataset construction. All metrics here measure the fine-tuned model's agreement with those GPT-5.4 references — not against gold human transcription.

> ⚠️ **A GPU with ≥24 GB VRAM is required** (same machine as training, or another GPU instance).

**Input:**
- `data/eval/eval.jsonl` — 280 eval samples (full set)
- `models/lora_adapter/run-2/` — adapter saved by Notebook 5

**Output:** `logs/run-2/eval_results.json`. Each evaluation writes into its run's log folder; switch `RUN_NAME` in the config cell to evaluate a different adapter (e.g. `run-1`).

## 1. Check GPU

In [1]:
!nvidia-smi

Sat Apr 18 02:04:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.16             Driver Version: 580.126.16     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10G                    On  |   00000000:00:1E.0 Off |                    0 |
|  0%   32C    P8             40W /  300W |       0MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Dependencies

In [2]:
import sys
!{sys.executable} -m pip install "transformers>=4.52.0" peft==0.14.0 "accelerate>=1.3.0" "bitsandbytes>=0.46.1" torchvision
!{sys.executable} -m pip install qwen-vl-utils jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 69.8 MB/s eta 0:00:0000:01


## 3. Set Project Root

In [3]:
from pathlib import Path

PROJECT_ROOT = Path('/home/ubuntu/nlp_project')

assert PROJECT_ROOT.exists(), f'Project root not found: {PROJECT_ROOT}'

print(f'Project root: {PROJECT_ROOT}')

Project root: /home/ubuntu/nlp_project


## 4. Configuration

In [ ]:
from pathlib import Path

BASE_MODEL = 'Qwen/Qwen2.5-VL-7B-Instruct'
MIN_PIXELS = 4   * 28 * 28   # must match training
MAX_PIXELS = 128 * 28 * 28   # must match training

RUN_NAME   = 'run-2'
CHECKPOINT = None  # None = merged final adapter; or e.g. 'checkpoint-1120'

ADAPTER_ROOT = Path(PROJECT_ROOT) / 'models' / 'lora_adapter' / RUN_NAME
ADAPTER_DIR  = ADAPTER_ROOT / CHECKPOINT if CHECKPOINT else ADAPTER_ROOT
EVAL_DATA    = Path(PROJECT_ROOT) / 'data' / 'eval' / 'eval.jsonl'
LOG_DIR      = Path(PROJECT_ROOT) / 'logs' / RUN_NAME

LOG_DIR.mkdir(parents=True, exist_ok=True)
assert ADAPTER_DIR.exists(), f'Adapter not found: {ADAPTER_DIR}'

print(f'Adapter: {ADAPTER_DIR}')
print(f'Eval:    {EVAL_DATA}')
print(f'Log dir: {LOG_DIR}')

## 5. Load Base Model + LoRA Adapter

In [5]:
import torch
from huggingface_hub import snapshot_download
from huggingface_hub.utils import LocalEntryNotFoundError
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from peft import PeftModel

HF_CACHE_DIR = None  # None = default ~/.cache/huggingface/hub/

# Verify disk cache — download only if missing
print(f'Verifying cache for: {BASE_MODEL}')
try:
    snapshot_path = snapshot_download(
        repo_id=BASE_MODEL, local_files_only=True, cache_dir=HF_CACHE_DIR,
    )
    print(f'  Cache OK — {snapshot_path}')
except LocalEntryNotFoundError:
    print(f'  Cache MISS — downloading {BASE_MODEL} (~16 GB)...')
    snapshot_path = snapshot_download(repo_id=BASE_MODEL, cache_dir=HF_CACHE_DIR)
    print(f'  Download complete.')

# Load base model weights into GPU memory from verified local path
print('Loading base model into GPU memory...')
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    snapshot_path,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='sdpa',
    local_files_only=True,
)

print('Loading LoRA adapter...')
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
model.eval()

processor = AutoProcessor.from_pretrained(
    snapshot_path,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    local_files_only=True,
)

print('Model and processor ready.')

/home/ubuntu/nlp_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Verifying cache for: Qwen/Qwen2.5-VL-7B-Instruct
  Cache OK — /home/ubuntu/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-7B-Instruct/snapshots/cc594898137f460bfe9f0759e9844b3ce807cfb5
Loading base model into GPU memory...


Loading weights:   0%|          | 2/729 [00:15<1:35:40,  7.90s/it]/home/ubuntu/nlp_project/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 729/729 [01:53<00:00,  6.40it/s]


Loading LoRA adapter...
Model and processor ready.


## 6. Define Metrics and JSON Parser

CER and WER are computed with `jiwer`, which clamps the edit distance to a proper rate in `[0, 1]` (the previous `difflib`-based scorer could exceed 1.0 on long mismatches). `parse_output()` recovers structured JSON from the model's raw text — handling markdown fences, truncated braces, and unterminated strings — so downstream metrics work even when the model drifts from the strict `{"transcription":"..."}` template.

In [6]:
import json
from jiwer import cer as _jiwer_cer, wer as _jiwer_wer

def parse_output(raw_text: str) -> dict:
    """Try multiple strategies to extract valid JSON."""
    text = raw_text.strip()

    # Strategy 1: Direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Strategy 2: Strip markdown code blocks
    if '```' in text:
        text = text.split('```json')[-1].split('```')[0].strip()
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass

    # Strategy 3: Find last closing brace
    for i in range(len(text) - 1, -1, -1):
        if text[i] == '}':
            try:
                return json.loads(text[:i+1])
            except json.JSONDecodeError:
                continue

    # Strategy 4: Auto-complete common truncations
    for suffix in ['"}',' "}', '"}', '"]}', '"}}']:
        try:
            return json.loads(text + suffix)
        except json.JSONDecodeError:
            continue

    return None


def char_error_rate(prediction: str, reference: str) -> float:
    if not reference:
        return 0.0 if not prediction else 1.0
    return float(_jiwer_cer(reference, prediction))


def word_error_rate(prediction: str, reference: str) -> float:
    if not reference.strip():
        return 0.0 if not prediction.strip() else 1.0
    return float(_jiwer_wer(reference, prediction))

print('Metric functions defined.')

Metric functions defined.


## 7. Run Evaluation

In [ ]:
import time
import base64
from io import BytesIO
from PIL import Image

def extract_images(messages):
    """Decode base64 data URLs from OpenAI-style `image_url` content into PIL Images."""
    images = []
    for msg in messages:
        content = msg.get('content')
        if not isinstance(content, list):
            continue
        for item in content:
            t = item.get('type')
            if t == 'image_url':
                url = item['image_url']
                if isinstance(url, dict):
                    url = url.get('url', '')
                if url.startswith('data:'):
                    b64 = url.split(',', 1)[1]
                    images.append(Image.open(BytesIO(base64.b64decode(b64))).convert('RGB'))
                elif url:
                    images.append(Image.open(url).convert('RGB'))
            elif t == 'image':
                src = item.get('image')
                if isinstance(src, Image.Image):
                    images.append(src)
                elif isinstance(src, str) and src.startswith('data:'):
                    b64 = src.split(',', 1)[1]
                    images.append(Image.open(BytesIO(base64.b64decode(b64))).convert('RGB'))
                elif isinstance(src, str):
                    images.append(Image.open(src).convert('RGB'))
    return images

# Load eval samples
eval_samples = []
with open(EVAL_DATA) as f:
    for line in f:
        eval_samples.append(json.loads(line))
print(f'Evaluating {len(eval_samples)} samples...')

# Build token suppression list (prevents <tool_call> hallucinations)
suppress_tokens = ['<tool_call>', '<|tool_call|>', '<tool_response>', '<|tool_response|>']
bad_words_ids = []
for token in suppress_tokens:
    ids = processor.tokenizer.encode(token, add_special_tokens=False)
    if ids:
        bad_words_ids.append(ids)

# Re-enable KV cache for inference (training disabled it for gradient checkpointing)
model.config.use_cache = True

eos_token_id = processor.tokenizer.eos_token_id
pad_token_id = processor.tokenizer.pad_token_id or eos_token_id

results = []
for i, sample in enumerate(eval_samples):
    messages = sample['messages']
    expected = messages[-1]['content']      # ground truth
    input_messages = messages[:-1]          # system + user

    text = processor.apply_chat_template(
        input_messages, tokenize=False, add_generation_prompt=True
    )

    # JSON prefix forcing — steers the model to produce valid JSON immediately
    json_prefix = '{"transcription":"'
    text += json_prefix

    # Decode base64 images directly (eval data uses OpenAI-style image_url format)
    image_inputs = extract_images(input_messages)

    inputs = processor(
        text=[text],
        images=image_inputs if image_inputs else None,
        padding=True,
        return_tensors='pt',
    ).to(model.device)

    start_time = time.time()
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,          # transcriptions are short (~5–20 words)
            do_sample=False,
            use_cache=True,              # KV cache on → big latency win
            repetition_penalty=1.1,
            no_repeat_ngram_size=6,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
            bad_words_ids=bad_words_ids if bad_words_ids else None,
        )
    inference_time = time.time() - start_time

    generated = processor.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    generated = json_prefix + generated  # re-attach prefix

    pred_parsed = parse_output(generated)
    ref_parsed  = parse_output(expected)

    pred_text = pred_parsed.get('transcription', '') if pred_parsed else ''
    ref_text  = ref_parsed.get('transcription',  '') if ref_parsed  else ''

    cer   = char_error_rate(pred_text, ref_text)
    wer   = word_error_rate(pred_text, ref_text)
    exact = pred_text.strip() == ref_text.strip()

    results.append({
        'cer': cer, 'wer': wer,
        'exact_match': exact,
        'valid_json': pred_parsed is not None,
        'inference_time': inference_time,
        'prediction': pred_text,
        'reference': ref_text,
    })

    print(f'[{i+1}/{len(eval_samples)}] CER={cer:.3f}  WER={wer:.3f}  Time={inference_time:.1f}s')

print('\nEvaluation complete.')

## 8. Summary & Save Results

In [ ]:
n = len(results)

avg_cer    = sum(r['cer']            for r in results) / n
avg_wer    = sum(r['wer']            for r in results) / n
exact_rate = sum(r['exact_match']    for r in results) / n * 100
json_rate  = sum(r['valid_json']     for r in results) / n * 100
avg_time   = sum(r['inference_time'] for r in results) / n

summary = {
    'num_samples':        n,
    'checkpoint':         CHECKPOINT or 'merged',
    'avg_cer':            round(avg_cer,    4),
    'avg_wer':            round(avg_wer,    4),
    'exact_match_rate':   round(exact_rate, 2),
    'valid_json_rate':    round(json_rate,  2),
    'avg_inference_time': round(avg_time,   2),
}

print('=== Evaluation Summary ===')
for k, v in summary.items():
    print(f'  {k}: {v}')

# Tag filename with checkpoint so multiple evals don't overwrite each other
suffix = f'_{CHECKPOINT}' if CHECKPOINT else ''
output_path = LOG_DIR / f'eval_results{suffix}.json'
with open(output_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\nResults saved to {output_path}')

## 9. Inspect Worst Predictions

In [ ]:
# Show the 5 samples with the highest CER
worst = sorted(results, key=lambda r: r['cer'], reverse=True)[:5]

print('=== Top 5 Highest CER Samples ===')
for i, r in enumerate(worst):
    print(f'\n#{i+1}  CER={r["cer"]:.3f}  WER={r["wer"]:.3f}')
    print(f'  Reference:  {r["reference"]}')
    print(f'  Prediction: {r["prediction"]}')

## Next Step (Optional)

If you want to serve the model as an API endpoint, continue to:

**Notebook 7 → `07_inference_server.ipynb`** — Launch a vLLM inference server